# Post Model Test Analysis (Part II)

Previosly on the "Post Model Test Analysis (Part II)" jupyter notebook, we demonstrated that our speaker age detection model did not perform well on the Speech Accent Archive dataset because the model was trained on short audio files (approximately 6 seconds long), and the audio samples from the Speech Accent Archive dataset tend to have much longer durations (approximately 20 seconds long).

This means that if we were to properly test our model on an audio ad, we will need to first split it by speaker segments, then make a prediction on each segment.

## Listen Demo Dataset

### Download Samples

A sample of 100 ads were taken from the listen demo dataset. A random seed of 42 was set for reproducibility.

In [1]:
import json
import random

with open('./datasets/listen_demo/listen_demo.json') as json_file:
    listen_demo_ads = json.load(json_file)
    
    # Added seed number for reproducibility
    random.seed(42)
    random.shuffle(listen_demo_ads)
    
    # Sample 100 Ads from listen_demo
    listen_demo_ads = listen_demo_ads[:100]

*listen_demo_ads* is a list of dictionaries, here is an example item from our list.

In [2]:
listen_demo_ads[50]

{'_id': {'$oid': '5ff34868769c9db6d37adea8'},
 'station': 'Anatomy of Murder',
 'timestamp': {'$date': {'$numberLong': '1608768464580'}},
 'start': '1369',
 'end': '1453',
 'basename': 'https://veritonic-ad-detection.s3.amazonaws.com/public/65e64b4d-1f58-4478-9835-0b1a1cbbe4e4.mp3',
 'last_detected': {'$date': {'$numberLong': '1608768464580'}},
 'brand': 'Squarespace',
 's3_file': 'demo/65e64b4d-1f58-4478-9835-0b1a1cbbe4e4_1369_1453.mp3',
 'duration': 84,
 'industry': 'B2B Services: Software & Solutions',
 'channel': 'podcast',
 'jobs': {'search_indexer': {'status': 'error',
   'version': 0,
   'finished_at': {'$date': {'$numberLong': '1609779903994'}},
   'started_at': {'$date': {'$numberLong': '1609779903992'}},
   'error_count': 6,
   'error_id': '8514027c-5759-46fe-a3bd-39863a361982',
   'error_message': "'ad_name'"}},
 'transcription': "Yeah, in today's landscape, whether you're launching a new podcast or start a business, or how about even a virtual storefront having a presence o

Now that we have our list of s3 links, where each link corresponds to an ad, we will use it to download our ads.

In [2]:
import requests

base_link = 'https://veritonic-ad-detection.s3.amazonaws.com/'
base_filename = './datasets/listen_demo/clips/%s'

for listen_demo_ad in listen_demo_ads:
    s3_file = listen_demo_ad['s3_file']
    s3_link = base_link + s3_file
    
    audio_response = requests.get(s3_link, allow_redirects=True)
    if audio_response.status_code == 200:
        filename = s3_file.split("/")[-1]
        current_filename = base_filename % filename
        
        with open(current_filename, 'wb') as audio_object:
            audio_object.write(audio_response.content)

### Apply Speaker Diarization to Downloaded Samples

Once we have downloaded our 100 ads, run the following scripts to get our speaker segments in the provided order.

1. *convert_mp3_files.py*
2. *apply_speaker_diarization.py*

*convert_mp3_files.py* takes in mp3 files and converts them to wav files. That way, our speaker diarization models can process our wav files instead of mp3 files.

### Generate Melspectrograms

For each speaker segment, generate melspectrograms.

In [3]:
import joblib
import numpy as np
import os
import warnings

from librosa import load, power_to_db
from librosa.feature import melspectrogram

def compute_melspectrogram(mp3_filepath):
    warnings.filterwarnings("ignore", category=UserWarning)
    y, sr = load(mp3_filepath)

    np_filename = mp3_filepath.split("/")[-1][:-4] + ".npy"
    np_filepath = './datasets/listen_demo/melspectrograms/%s' % np_filename

    melspectrogram_db = melspectrogram(y, sr)
    melspectrogram_db = power_to_db(melspectrogram_db, ref=np.max)
    melspectrogram_db = np.resize(melspectrogram_db.T, (431, 128)).T
    melspectrogram_db = melspectrogram_db.reshape(
        *melspectrogram_db.shape, 1
    )
    melspectrogram_db = (melspectrogram_db + 80) / 80
    
    np.save(np_filepath, melspectrogram_db)

mp3_directory = './datasets/listen_demo/speaker_segments/'
mp3_filenames = os.listdir(mp3_directory)
mp3_filepaths = [
    mp3_directory + mp3_filename
    for mp3_filename in mp3_filenames
]

jobs = [
    joblib.delayed(compute_melspectrogram)(mp3_filepath)
    for mp3_filepath in mp3_filepaths
]
        
result = joblib.Parallel(n_jobs=12, verbose=1)(jobs)

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  30 tasks      | elapsed:    3.2s
[Parallel(n_jobs=12)]: Done 209 out of 209 | elapsed:   17.4s finished


### Predict Speaker's Age Group

In [17]:
from tensorflow.keras.models import load_model

model = load_model('models/hdf5/model12/model12-04.hdf5')
labels = ["under-20", "20-29", "30-39", "40-49", "50-59", "60+"]

# Melspectrograms
directory = './datasets/listen_demo/melspectrograms/'
filenames = os.listdir(directory)
filepaths = [directory + filename for filename in filenames]

# Predicted Age Group
predictions = []
for filepath in filepaths:
    melspectrogram = np.load(filepath)
    melspectrogram = np.array([melspectrogram])
    
    prediction = model.predict(melspectrogram)
    prediction = prediction.argmax(axis=1)[0]
    prediction = labels[prediction]
    
    predictions.append(prediction)

### Consolidate Listen Demo, and Predictions into Dataframes

In [28]:
import pandas as pd

#### Listen Demo

In [32]:
brands = []
s3_files = []
channels = []
durations = []
industries = []

for listen_demo_ad in listen_demo_ads:
    brand = listen_demo_ad['brand']
    s3_file = listen_demo_ad['s3_file'].split("/")[-1][:-4]
    channel = listen_demo_ad['channel']
    duration = listen_demo_ad['duration']
    industry = listen_demo_ad['industry']
    
    brands.append(brand)
    s3_files.append(s3_file)
    channels.append(channel)
    durations.append(duration)
    industries.append(industry)
    
listen_demo_ads_df = pd.DataFrame(
    {
        's3_file': s3_files,
        'brand': brands,
        'duration': durations,
        'industry': industries,
        'channel': channels
    }
)
listen_demo_ads_df = listen_demo_ads_df.sort_values(by='s3_file')
listen_demo_ads_df.head(10)

,s3_file,brand,duration,industry,channel
92,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,FanDuel,84.0,Entertainment: Gaming,podcast
15,0ad42c62a5775d37b2dbcf2a25c41655_50_81,Epic Will,31.0,Professional Services: Legal,podcast
35,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,Noom,73.0,Fitness & Nutrition,podcast
24,103a5ae1-e663-4f6c-8ab9-65d0e695694b.original_...,We the People Holsters,96.0,Retail,podcast
81,14ee459c8ba659ad88892536458830a5_1540_1645.5,BetMGM,105.5,Entertainment: Gaming,podcast
4,1da75aac-5d6d-4bfa-8b0f-3bcb65a9ab8d.original_...,ThirdLove,140.0,Retail: Apparel & Accessories,podcast
91,25996dbc36035ccdaa729b89cfb32dc9_0_30,Kubota,30.0,Technology,podcast
48,2ba8073a80a7516991da1c4dbb97972e_119.5_159,Indeed,39.5,B2B Services: Career,podcast
40,3d7f4a9a-7caa-4422-9d4d-e985ee98d288_286_343,Discovery+,57.0,Entertainment: Streaming Services,podcast
83,48021d396354537fa38e2ee2bea1cb24_488_518,Free People,30.0,Retail: Apparel & Accessories,podcast


#### Predictions

In [82]:
import re

s3_files = []
speaker_ids = []
segment_indices = []

npy_segment_files = []
mp3_segment_files = []

for filepath in filepaths:
    s3_file = re.sub('_\d_SPEAKER_\d\d', "", filepath.split("/")[-1][:-5])
    speaker_id = re.findall('SPEAKER_\d\d', filepath)[0]
    segment_index = re.findall('_\d_SPEAKER_\d\d', filepath)[0].split("_")[1]
    
    npy_segment_file = filepath.split("/")[-1]
    mp3_segment_file = npy_segment_file[:-4] + '.mp3'
    
    s3_files.append(s3_file)
    speaker_ids.append(speaker_id)
    segment_indices.append(segment_index)
    
    npy_segment_files.append(npy_segment_file)
    mp3_segment_files.append(mp3_segment_file)

predictions_df = pd.DataFrame(
    {
        's3_file': s3_files,
        'segment_index': segment_indices,
        'npy_segment_file': npy_segment_files,
        'mp3_segment_file': mp3_segment_files,
        'speaker_id': speaker_ids,
        'speaker_age': predictions
    }
)
predictions_df = predictions_df.sort_values(by=['s3_file', 'segment_index'])
predictions_df.head(10)

,s3_file,segment_index,npy_segment_file,mp3_segment_file,speaker_id,speaker_age
152,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,0,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,SPEAKER_00,50-59
6,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,1,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,09519757-d650-48e1-b57b-8c9aa96f2c93.original_...,SPEAKER_00,20-29
119,0ad42c62a5775d37b2dbcf2a25c41655_50_81,0,0ad42c62a5775d37b2dbcf2a25c41655_50_81_0_SPEAK...,0ad42c62a5775d37b2dbcf2a25c41655_50_81_0_SPEAK...,SPEAKER_00,40-49
29,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_01,20-29
17,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,1,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_00,20-29
78,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,2,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_01,20-29
146,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,3,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_00,20-29
91,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,4,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_01,20-29
39,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,5,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_00,20-29
73,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,6,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,0b5b0762-24f7-4975-bba2-7892e5d94b7a.original_...,SPEAKER_01,40-49


#### Save Results

In [83]:
listen_demo_ads_df.to_csv("listen_demo_ads.csv", index=False)
predictions_df.to_csv("predictions.csv", index=False)

### Consolidate Listen Demo, and Predictions into a Json File

Each entry in the json list has an s3_file, and this can be used to get the s3_link (see comments below for more info).

In [102]:
# Template: https://veritonic-ad-detection.s3.amazonaws.com/demo/' + s3_file + '.mp3'
import json

outputs = []
for _, listen_demo_ad in listen_demo_ads_df.iterrows():
    s3_file = listen_demo_ad['s3_file']
    output = {
        's3_file': s3_file,
        'brand': listen_demo_ad['brand'],
        'duration': listen_demo_ad['duration'],
        'industry': listen_demo_ad['industry'],
        'channel': listen_demo_ad['channel']
    }
    output['segments'] = []
 
    segments_info_df = predictions_df[predictions_df.s3_file == s3_file]
    for _, segment_info in segments_info_df.iterrows():
        output['segments'].append(
            {
                'segment_index': segment_info['segment_index'],
                'npy_segment_file': segment_info['npy_segment_file'],
                'mp3_segment_file': segment_info['mp3_segment_file'],
                'speaker_id': segment_info['speaker_id'],
                'speaker_age': segment_info['speaker_age'],
            }
        )
    
    outputs.append(output)

with open('predictions.json', 'w') as json_file:
    json.dump(outputs, json_file, indent=2)

**Note: CSV files, audio files, and the Json file are available upon request.**